In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/rapido_project/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/rapido_project/notebooks', exist_ok=True)

Mounted at /content/drive


In [2]:
!pip install duckdb plotly shap xgboost imbalanced-learn

In [3]:


import pandas as pd

rides = pd.read_csv('/content/drive/MyDrive/rapido_project/data/rides_data.csv')

cancel_rates = rides.groupby('services')['ride_status'].apply(
    lambda x: (x == 'cancelled').mean()
).round(3)
print("Cancellation rates:\n", cancel_rates)

rides_completed = rides[rides['ride_status'] == 'completed'].copy()
rides_completed['fare_per_km'] = rides_completed['total_fare'] / rides_completed['distance']
fare_stats = rides_completed.groupby('services')['fare_per_km'].mean().round(2)
print("\nFare per km:\n", fare_stats)

duration_stats = rides_completed.groupby('services')['duration'].mean().round(1)
print("\nAvg duration (mins):\n", duration_stats)

payment_split = rides['payment_method'].value_counts(normalize=True).round(3)
print("\nPayment methods:\n", payment_split)

Cancellation rates:
 services
auto           0.098
bike           0.103
bike lite      0.102
cab economy    0.103
parcel         0.095
Name: ride_status, dtype: float64

Fare per km:
 services
auto           44.16
bike           43.99
bike lite      43.38
cab economy    44.07
parcel         42.46
Name: fare_per_km, dtype: float64

Avg duration (mins):
 services
auto           65.0
bike           64.0
bike lite      64.0
cab economy    64.2
parcel         64.4
Name: duration, dtype: float64

Payment methods:
 payment_method
Paytm         0.252
GPay          0.251
Amazon Pay    0.250
QR scan       0.248
Name: proportion, dtype: float64


In [5]:
print(df.columns.tolist())
print(df.shape)

NameError: name 'df' is not defined

In [ ]:
import numpy as np
import pandas as pd
import duckdb

np.random.seed(42)
N = 10000

# ============================================================
# SECTION 1 — CAPTAIN IDENTITY
# ============================================================

cities = ['Bengaluru', 'Hyderabad', 'Chennai', 'Pune', 'Delhi']
city_weights = [0.35, 0.25, 0.18, 0.12, 0.10]

df = pd.DataFrame()
df['captain_id'] = [f'CAP{str(i).zfill(5)}' for i in range(N)]
df['city'] = np.random.choice(cities, N, p=city_weights)
df['vehicle_type'] = np.random.choice(['bike', 'auto'], N, p=[0.72, 0.28])
df['signup_month'] = np.random.choice(range(1, 13), N)

df['captain_type'] = np.random.choice(
    ['fulltime', 'parttime'], N, p=[0.40, 0.60]
)

fulltime_mask = df['captain_type'] == 'fulltime'
bike_mask = df['vehicle_type'] == 'bike'

# ============================================================
# SECTION 2 — RIDES PER WEEK (decay pattern)
# Fulltime: 10-15 rides/day → ~80/week
# Parttime: 4-6 rides/day  → ~28/week
# Source: Reddit r/bangalore captain posts
# ============================================================

df['rides_week1'] = np.where(
    fulltime_mask,
    np.random.poisson(80, N).clip(40, 120),
    np.random.poisson(28, N).clip(5, 50)
)
df['rides_week2'] = (df['rides_week1'] * np.random.uniform(0.55, 0.95, N)).astype(int)
df['rides_week3'] = (df['rides_week2'] * np.random.uniform(0.45, 0.95, N)).astype(int)
df['rides_week4'] = (df['rides_week3'] * np.random.uniform(0.35, 0.95, N)).astype(int)

df['ride_decay_rate'] = (
    (df['rides_week1'] - df['rides_week4']) / (df['rides_week1'] + 1)
).round(3)

# ============================================================
# SECTION 3 — TIME PATTERNS
# Night rides: Rapido gives 20% bonus 10PM-6AM
# Source: Reddit r/bangalore captain experience post 2024
# Parttime captains lean more toward nights (side hustle behavior)
# ============================================================

df['night_rides_ratio'] = np.where(
    fulltime_mask,
    np.random.beta(2, 3, N),
    np.random.beta(3, 2, N)
).round(3)

df['peak_hour_ratio'] = np.random.beta(2, 3, N).round(3)

# ============================================================
# SECTION 4 — RIDE QUALITY METRICS
# Cancellation: bike ~12-18%, auto ~4-14%
# Duration: bike shorter (17 mins avg), auto longer (24 mins avg)
# Source: Industry reports + Reddit vlogs
# ============================================================

df['cancellation_rate'] = np.where(
    bike_mask,
    np.random.beta(2, 5, N).clip(0.08, 0.22),
    np.random.beta(1.5, 7, N).clip(0.04, 0.14)
).round(3)

df['avg_ride_duration'] = np.where(
    bike_mask,
    np.random.normal(17, 4, N).clip(8, 35),
    np.random.normal(24, 6, N).clip(10, 45)
).round(1)

# ============================================================
# SECTION 5 — EARNINGS
# Bike fare: ₹8-16/km, Auto fare: ₹13-26/km
# Source: Reddit captain vlogs, common knowledge
# Petrol cost sensitivity — proxy for thin margin pressure
# ============================================================

df['avg_fare_per_km'] = np.where(
    bike_mask,
    np.random.normal(11, 2, N).clip(7, 16),
    np.random.normal(19, 3, N).clip(13, 26)
).round(2)

df['petrol_cost_sensitivity'] = np.random.beta(2, 3, N).round(3)

df['estimated_daily_earnings'] = (
    (df['rides_week4'] / 6) * 
    df['avg_fare_per_km'] * 8 * 
    (1 - df['cancellation_rate'])
).round(0)

# ============================================================
# SECTION 6 — ZONE & NAVIGATION BEHAVIOR
# Experienced captains stick to high-density zones
# Churning captains switch desperately
# Source: Reddit: "pick high-density routes not short quick rides"
# ============================================================

df['zone_switches'] = np.where(
    fulltime_mask,
    np.random.poisson(1.5, N).clip(0, 5),
    np.random.poisson(3.0, N).clip(0, 10)
).astype(int)

# ============================================================
# SECTION 7 — INCENTIVE & ENGAGEMENT
# Streak completion ~27% — "8 rides in 4hrs is hard"
# Source: Reddit r/bangalore captain post 2024
# ============================================================

df['incentive_claimed'] = np.random.binomial(1, 0.52, N)
df['streak_completed'] = np.random.binomial(1, 0.27, N)
df['app_opens_per_day'] = np.random.poisson(6, N).clip(1, 20).astype(int)
df['support_tickets'] = np.random.poisson(0.9, N).clip(0, 5).astype(int)

# ============================================================
# SECTION 8 — PAYMENT METHOD
# GPay dominant post-2023 (~40%)
# Paytm dropped after RBI action (~20%)
# Source: NPCI UPI market share report 2024
# ============================================================

df['payment_method_pref'] = np.random.choice(
    ['gpay', 'paytm', 'amazon_pay', 'qr_scan', 'cash'],
    N,
    p=[0.40, 0.20, 0.18, 0.12, 0.10]
)

# ============================================================
# SECTION 9 — CHURN LABEL (TARGET VARIABLE)
# Logic grounded in real dropout reasons:
# - Rapid ride decay (biggest signal)
# - High cancellation rate
# - No incentive claimed
# - Low earnings per km
# - High zone switching (desperation)
# - Support complaints
# - Parttime type (higher baseline churn)
# ============================================================

churn_score = (
    - 0.15 * df['rides_week4'].clip(0, 20)
    + 0.35 * df['cancellation_rate'] * 10
    - 0.20 * df['incentive_claimed']
    - 0.15 * df['streak_completed']
    + 0.20 * df['support_tickets']
    + 0.15 * df['zone_switches']
    + 0.10 * df['petrol_cost_sensitivity'] * 5
    + 0.10 * (df['captain_type'] == 'parttime').astype(int)
    - 0.05 * (df['avg_fare_per_km'] - 11)
    + 0.20 * df['ride_decay_rate']
    + 0.15
)

churn_prob = 1 / (1 + np.exp(-churn_score))
df['is_churned'] = np.random.binomial(1, churn_prob)

# Verify
print(f"Churn rate: {df['is_churned'].mean():.2%}  ← should be 35-45%")

# ============================================================
# SECTION 10 — VALIDATION
# ============================================================

print("=" * 50)
print("VALIDATION CHECKS")
print("=" * 50)

print(f"\nOK - Shape: {df.shape}  ← should be (10000, 23)")

churn_rate = df['is_churned'].mean()
status = "OK -" if 0.35 <= churn_rate <= 0.45 else "⚠️ RECHECK"
print(f"{status} Churn rate: {churn_rate:.2%}  ← target: 35-45%")

print(f"\nOK - Ride decay (avg per week):")
print(df[['rides_week1','rides_week2','rides_week3','rides_week4']].mean().round(1))

print(f"\nOK - Churned vs Active — Week 4 rides:")
print(df.groupby('is_churned')['rides_week4'].mean().round(1))

print(f"\nOK - Churn by captain type:")
print(df.groupby('captain_type')['is_churned'].mean().round(2))

print(f"\nOK - Churn by vehicle type:")
print(df.groupby('vehicle_type')['is_churned'].mean().round(2))

print(f"\nOK - Churn by city:")
print(df.groupby('city')['is_churned'].mean().round(2).sort_values(ascending=False))

print(f"\nOK - Avg fare per km by vehicle:")
print(df.groupby('vehicle_type')['avg_fare_per_km'].mean().round(2))

print(f"\nOK - Cancellation rate by vehicle:")
print(df.groupby('vehicle_type')['cancellation_rate'].mean().round(3))

print(f"\nOK - Nulls: {df.isnull().sum().sum()}  ← should be 0")
from google.colab import drive
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/rapido_project/data/captain_synthetic.csv'
df.to_csv(drive_path, index=False)
print("\n✅ Saved as captains_synthetic.csv")
print("\nColumn list:")
print(df.columns.tolist())

Churn rate: 41.06%  ← should be 35-45%
VALIDATION CHECKS

OK - Shape: (10000, 24)  ← should be (10000, 23)
OK - Churn rate: 41.06%  ← target: 35-45%

OK - Ride decay (avg per week):
rides_week1    48.8
rides_week2    36.0
rides_week3    24.7
rides_week4    15.5
dtype: float64

OK - Churned vs Active — Week 4 rides:
is_churned
0    19.1
1    10.5
Name: rides_week4, dtype: float64

OK - Churn by captain type:
captain_type
fulltime    0.19
parttime    0.56
Name: is_churned, dtype: float64

OK - Churn by vehicle type:
vehicle_type
auto    0.33
bike    0.44
Name: is_churned, dtype: float64

OK - Churn by city:
city
Bengaluru    0.42
Chennai      0.42
Pune         0.41
Delhi        0.40
Hyderabad    0.40
Name: is_churned, dtype: float64

OK - Avg fare per km by vehicle:
vehicle_type
auto    19.04
bike    11.01
Name: avg_fare_per_km, dtype: float64

OK - Cancellation rate by vehicle:
vehicle_type
auto    0.111
bike    0.188
Name: cancellation_rate, dtype: float64

OK - Nulls: 0  ← should be 0